# Amy Njenga - 668698
## DSA 4060 Quiz 1
### Assigned UserID = 19

### Importing Libraries


In [1]:
import pandas as pd
import matplotlib.pyplot as plt



### Loading and Inspecting the Data

• Display the first five rows of each dataset. (1 mark)

• Show the number of rows and columns in each dataset. (1 mark)

• Check for missing values in both datasets and briefly state what you find. (1 mark)

In [2]:
movies = pd.read_csv(r"C:\Users\jkaru\dsa4060-week5-quiz\data\movies.csv")
ratings = pd.read_csv(r"C:\Users\jkaru\dsa4060-week5-quiz\data\ratings.csv")

In [3]:
print("Movies shape: ", movies.shape)
print("Ratings shape: ", ratings.shape)

display(movies.head())
display (ratings.head())

display(movies.info())
display(ratings.info())

print("Missing Values in Movies: ")
print(movies.isnull().sum())
print("Missing Values in Ratings: ")
print (ratings.isnull().sum())



Movies shape:  (9742, 3)
Ratings shape:  (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


None

Missing Values in Movies: 
movieId    0
title      0
genres     0
dtype: int64
Missing Values in Ratings: 
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


From the output above it shows that both datasets contain no missing values in any of the columns, this is means we can proceed with the analysis

### Profiling the assigned User

• **List all movies rated by the user, including title, genres, and rating. (2 marks)**

• **Identify the user’s three highest-rated movies. If there is a tie, show any valid three from the tied highest ratings. (1 mark)**

• **Determine the user’s apparent preferred genre or genres. Use their ratings as evidence and explain your reasoning in 2–4 sentences. (2 marks)**

In [4]:

movie_stats = ratings.loc[ratings["userId"].eq(19)].merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="many_to_one"
)
display(movie_stats)

,userId,movieId,rating,timestamp,title,genres
0,19,1,4.0,965705637,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,19,2,3.0,965704331,Jumanji (1995),Adventure|Children|Fantasy
2,19,3,3.0,965707636,Grumpier Old Men (1995),Comedy|Romance
3,19,7,2.0,965706657,Sabrina (1995),Comedy|Romance
4,19,10,2.0,965709556,GoldenEye (1995),Action|Adventure|Thriller
...,...,...,...,...,...,...
698,19,3825,3.0,965702777,Coyote Ugly (2000),Comedy|Drama|Romance
699,19,3826,1.0,965702843,Hollow Man (2000),Horror|Sci-Fi|Thriller
700,19,3835,2.0,965711413,"Crush, The (1993)",Thriller
701,19,3837,1.0,965704999,Phantasm II (1988),Action|Fantasy|Horror|Sci-Fi|Thriller


In [5]:
display(movie_stats.sort_values(by="rating", ascending=False).head(3))

,userId,movieId,rating,timestamp,title,genres
354,19,1967,5.0,965704281,Labyrinth (1986),Adventure|Fantasy|Musical
638,19,3477,5.0,965706839,Empire Records (1995),Comedy|Drama
274,19,1527,5.0,965703811,"Fifth Element, The (1997)",Action|Adventure|Comedy|Sci-Fi


The top 3 rated movies for our specific user are Labyrinth (1986), Empire Records(1995), Fifth Element, The (1997) all of which are rated 5 stars.

The user's apparent preferred genres are *Adventure* and *Comedy*, both of which appear multiple times across the 3 movies. Labyrinth is an Adventure + Fantasy while Fifth Element is an Action + Adventure. This shows some overlap in the Adventure genre. Empire Records is a Comedy + Drama while Fifth Element is an Adventure + Comedy. This shows some overlap as well with the Comedy genre.

**Create a simple content-based recommendation process using movie genres. Your approach should use the user’s positively rated movies as evidence of preference.**

• **Treat movies rated 4.0 or higher as movies the user likes. (1 mark)**

• **Convert the genres into numerical features using a suitable text/vector representation such as CountVectorizer or TF-IDF. (2 marks)**

• **Use cosine similarity, or an equivalent similarity measure, to identify movies similar to those the user likes.**
(2 marks)

• **Exclude movies already rated by the assigned user and return exactly five recommended movie titles. (2 marks)**

In [6]:
rated_features = movie_stats["genres"]
user_ratings = movie_stats["rating"]

display(rated_features)
display(user_ratings)

0      Adventure|Animation|Children|Comedy|Fantasy
1                       Adventure|Children|Fantasy
2                                   Comedy|Romance
3                                   Comedy|Romance
4                        Action|Adventure|Thriller
                          ...                     
698                           Comedy|Drama|Romance
699                         Horror|Sci-Fi|Thriller
700                                       Thriller
701          Action|Fantasy|Horror|Sci-Fi|Thriller
702                                         Horror
Name: genres, Length: 703, dtype: str

0      4.0
1      3.0
2      3.0
3      2.0
4      2.0
      ... 
698    3.0
699    1.0
700    2.0
701    1.0
702    1.0
Name: rating, Length: 703, dtype: float64

### One Hot Encoding of Features

In [14]:
liked_movies = movie_stats[movie_stats["rating"] >= 4.0].copy()


weighted_features = (
    liked_movies["genres"]
    .str.get_dummies(sep="|")
    .mul(liked_movies["rating"].to_numpy(), axis=0)
)
weighted_features

,Action,Adventure,Animation,Children,Comedy,Crime,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War
0,0.0,4.0,4.0,4.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,4.0,0.0
10,0.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
638,0.0,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
685,4.0,4.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
688,0.0,4.0,4.0,4.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Creation of the Raw Aggregate user profile

In [8]:
user_profile = weighted_features.sum()
user_profile

Action       119.0
Adventure    126.0
Animation     58.0
Children      84.0
Comedy       235.0
Crime         76.0
Drama        101.0
Fantasy       96.0
Film-Noir     13.0
Horror        20.0
Musical       39.0
Mystery       65.0
Romance       76.0
Sci-Fi       103.0
Thriller     107.0
War            8.0
dtype: float64

In [9]:
user_profile.sort_values( ascending=False )

Comedy       235.0
Adventure    126.0
Action       119.0
Thriller     107.0
Sci-Fi       103.0
Drama        101.0
Fantasy       96.0
Children      84.0
Romance       76.0
Crime         76.0
Mystery       65.0
Animation     58.0
Musical       39.0
Horror        20.0
Film-Noir     13.0
War            8.0
dtype: float64

### Normalise the user profile

In [10]:
user_profile_normalized = ( user_profile / user_profile.sum() ) 
user_profile_normalized

Action       0.089744
Adventure    0.095023
Animation    0.043741
Children     0.063348
Comedy       0.177225
Crime        0.057315
Drama        0.076169
Fantasy      0.072398
Film-Noir    0.009804
Horror       0.015083
Musical      0.029412
Mystery      0.049020
Romance      0.057315
Sci-Fi       0.077677
Thriller     0.080694
War          0.006033
dtype: float64

### Caculation of similarity scores using dot product between normalised vector and binary genre vector

In [17]:
candidate_genre_matrix = (
    movies["genres"]
    .fillna("")
    .str.get_dummies(sep="|")
    .reindex(columns=user_profile_normalized.index, fill_value=0)
)

movies["recommendation_score"] = candidate_genre_matrix.mul(
    user_profile_normalized, axis=1
).sum(axis=1)

In [18]:
ranked_movies = movies.sort_values( "recommendation_score", ascending=False ) 
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
7441,Rubber (2010),0.650075
4631,Interstate 60 (2002),0.628205
6462,Aqua Teen Hunger Force Colon Movie Film for Th...,0.604827
9394,Maximum Ride (2016),0.592760
8597,Dragonheart 2: A New Beginning (2000),0.591252
...,...,...
8919,Bruce Lee: A Warrior's Journey (2000),0.000000
4200,Lost in La Mancha (2002),0.000000
4218,Sunless (Sans Soleil) (1983),0.000000
2702,Better Living Through Circuitry (1999),0.000000


In [16]:
watched_movies = ratings.loc[ratings["userId"] == 19, "movieId"]
recommendations = ranked_movies[~ranked_movies["movieId"].isin(watched_movies)]
recommendations[["title", "recommendation_score", "genres"]].head(5)


,title,recommendation_score,genres
7441,Rubber (2010),0.650075,Action|Adventure|Comedy|Crime|Drama|Film-Noir|...
4631,Interstate 60 (2002),0.628205,Adventure|Comedy|Drama|Fantasy|Mystery|Sci-Fi|...
6462,Aqua Teen Hunger Force Colon Movie Film for Th...,0.604827,Action|Adventure|Animation|Comedy|Fantasy|Myst...
9394,Maximum Ride (2016),0.592760,Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller
8597,Dragonheart 2: A New Beginning (2000),0.591252,Action|Adventure|Comedy|Drama|Fantasy|Thriller


#### Rubber (2010) 

This is the highest overall match to the user's preferences, It correspnds to Comedy, Adventure and Action having the highest contribution towards the user's preference profile. This film has strong elements of all three genres among others such as Crime, Drama and Film-Noir.

#### Interstate 60 (2002) 

This is not similar to Rubber as it contains the Action + Comedy genre preference, but it also represents the user's affinity for Fantasy elements.

#### Aqua Teen Hunger Force

This contains almost a complete overlap with the user's genre preferences which are Action + Adventure + Comedy + Fantasy.

#### Maximum Ride

This contains the Action + Adventure + Comedy combination as well as some fantasy elements which also contributed to the preference profile.

#### Dragon Heart

This contains similar genres to the rest of the recommendations and aligns closely with the user's positively rated movies.


## Limitations

The current recommender system may fail to accurately capture a user's preference. The system may recommend movies that are extremely similar to what the user last watched without correctly identifying the feature that caused them to rate the movie highly. This is due to relying on surface level similarities between the movies. This may reduce the quality of the recommendations.

In this recommeder system, the features being considered are the movie genres, but a user may have rated a movie highly due to the soundtrack, actor, director etc. 

## Improvement

One way to make the recommendations more accurate would be to expand the features used to determine the user preference profile which would then influence the reccomendation score and thus the recommendations. This could be achieved by incorporating the metadata of the movies
